In [1]:
import pandas as pd

class PWaveArrivalLocator:
    def __init__(self, csv_path):
        """
        初始化：只在建立物件時讀取一次 CSV 檔案，節省 I/O 時間。
        """
        print(f"[*] 正在初始化並載入 CSV 檔案: {csv_path} ...")
        self.df = pd.read_csv(csv_path, low_memory=False)
        
        # 檢查必備欄位是否存在
        required_cols = ['source_event_id', 'station_code', 'trace_p_arrival_sample']
        for col in required_cols:
            if col not in self.df.columns:
                raise ValueError(f"[!] CSV 檔案中缺少必要的欄位: {col}")
                
        # 為了確保 .str.startswith() 可以正常運作，將事件 ID 欄位強制轉為字串
        self.df['source_event_id'] = self.df['source_event_id'].astype(str)
        print("[+] CSV 載入完成，查詢器準備就緒！")

    def get_arrival_time(self, hdf5_key):
        """
        輸入: HDF5 的檔名 (例如 '2021_2112310800_YUS_...')
        輸出: 對應的 trace_p_arrival_sample 值 (整數)。若找不到則回傳 None。
        """
        try:
            # 1. 解析 HDF5 檔名
            parts = hdf5_key.split('_')
            if len(parts) < 3:
                return None
                
            event_id = parts[1]      # 取得 '2112310800'
            station_code = parts[2]  # 取得 'YUS'
            
        except Exception:
            return None

        # 2. 雙重條件篩選
        # 條件 A: source_event_id 開頭包含 HDF5 解析出來的 event_id
        # 條件 B: station_code 完全符合
        condition = (self.df['source_event_id'].str.startswith(event_id)) & \
                    (self.df['station_code'] == station_code)
                    
        # 3. 取得符合條件的資料列
        matched_row = self.df[condition]
        
        # 4. 回傳結果
        if not matched_row.empty:
            # 假設條件足以篩選出唯一一筆，我們取第一筆的數值 (.iloc[0])
            arrival_sample = matched_row['trace_p_arrival_sample'].iloc[0]
            
            # 確保回傳的是乾淨的整數，不要帶有 NaN
            if pd.isna(arrival_sample):
                return None
            return int(arrival_sample)
        else:
            # 找不到符合的資料
            return None

# ==========================================
# 測試區塊 (模擬你丟入 HDF5 檔名的情境)
# ==========================================
if __name__ == "__main__":
    # 假設這是你的 CSV 檔案路徑
    csv_file = r'Y:\CWA_processed_data\all_metadata.csv'
    
    try:
        # 實例化這個 Class (此時會讀取 CSV)
        locator = PWaveArrivalLocator(csv_file)
        
        # 模擬從 h5py 抓出來的檔名
        test_hdf5_keys = [
            "2021_2112310800_EHY_HL_SMT_10",
            "2021_2112310800_ENA_HL_SMT_10", # 假設這筆存在
            "2021_2112310800_ENT_HL_SMT_10"  # 假設這筆是亂碼找不到
        ]
        
        print("\n--- 開始測試查詢 ---")
        for key in test_hdf5_keys:
            p_arrival = locator.get_arrival_time(key)
            if p_arrival is not None:
                print(f"✅ 檔名: {key} \n   -> 找到 P波位置: {p_arrival}")
            else:
                print(f"❌ 檔名: {key} \n   -> 找不到對應的資料")
                
    except FileNotFoundError:
        print(f"[!] 測試失敗：找不到 {csv_file}，請確認路徑。")

[*] 正在初始化並載入 CSV 檔案: Y:\CWA_processed_data\all_metadata.csv ...
[+] CSV 載入完成，查詢器準備就緒！

--- 開始測試查詢 ---
✅ 檔名: 2021_2112310800_EHY_HL_SMT_10 
   -> 找到 P波位置: 3614
✅ 檔名: 2021_2112310800_ENA_HL_SMT_10 
   -> 找到 P波位置: 3186
✅ 檔名: 2021_2112310800_ENT_HL_SMT_10 
   -> 找到 P波位置: 3618


In [ ]:
import h5py
import os

def create_sample_hdf5(source_path, output_path, sample_size=1000):
    """
    從巨大的 HDF5 檔案中，按順序抽取指定數量的資料，另存為小檔案。
    """
    print(f"[*] 準備從 {source_path} 建立抽樣檔案...")
    
    if not os.path.exists(source_path):
        print(f"[!] 錯誤：找不到來源檔案 {source_path}")
        return

    try:
        with h5py.File(source_path, 'r') as src, h5py.File(output_path, 'w') as dst:
            
            # 1. 取得所有資料的 Key (檔名)
            # 注意：如果檔案真的極大，list(src.keys()) 可能會稍微停頓一下，請耐心等候幾秒
            all_keys = list(src.keys())
            total_records = len(all_keys)
            print(f"[+] 成功讀取目錄！原始檔案共有 {total_records} 筆資料。")
            
            # 2. 決定要抽取的範圍 (避免要求的數量大於實際總數)
            actual_size = min(sample_size, total_records)
            target_keys = all_keys[:actual_size]
            
            print(f"[*] 開始複製前 {actual_size} 筆資料到 {output_path} ...")
            
            # 3. 執行複製
            for i, key in enumerate(target_keys, 1):
                src.copy(key, dst)
                
                # 每複製 100 筆印出一次進度，避免畫面狂刷反而拖慢速度
                if i % 100 == 0 or i == actual_size:
                    print(f"    進度: {i} / {actual_size} (已複製: {key})")
                    
        print("-" * 40)
        print(f"✅ 抽樣完成！已經成功建立包含 {actual_size} 筆資料的小型 HDF5 檔案。")
        print(f"📁 檔案位置: {output_path}")
        print("-" * 40)

    except Exception as e:
        print(f"[!] 處理時發生錯誤: {e}")

if __name__ == "__main__":
    # --- 路徑設定 ---
    # NAS 上的大檔案路徑
    huge_hdf5_file = r'Y:\CWA_processed_data\all.hdf5' 
    
    # 輸出在當前資料夾的小檔案名稱
    sample_hdf5_file = 'sample_3000.hdf5' 
    
    # 執行抽樣 (預設 1000 筆)
    create_sample_hdf5(huge_hdf5_file, sample_hdf5_file, sample_size=3000)

[*] 準備從 Y:\CWA_processed_data\all.hdf5 建立抽樣檔案...
[+] 成功讀取目錄！原始檔案共有 465366 筆資料。
[*] 開始複製前 1000 筆資料到 sample_1000.hdf5 ...
    進度: 100 / 1000 (已複製: 2012_1201011423_CHK_HL_SMT_10)
    進度: 200 / 1000 (已複製: 2012_1201040653_ELD_HL_BB_10)
    進度: 300 / 1000 (已複製: 2012_1201040659_SGL_HL_SMT_10)
    進度: 400 / 1000 (已複製: 2012_1201041324_ENA_HL_SMT_10)
    進度: 500 / 1000 (已複製: 2012_1201050229_ELD_HL_BB_10)
    進度: 600 / 1000 (已複製: 2012_1201051632_SML_HL_SMT_10)
    進度: 700 / 1000 (已複製: 2012_1201090340_TWL_HL_SMT_10)
    進度: 800 / 1000 (已複製: 2012_1201090616_ESL_HL_SMT_10)
    進度: 900 / 1000 (已複製: 2012_1201091326_YUS_HL_SMT_10)
    進度: 1000 / 1000 (已複製: 2012_1201100800_ELD_HL_BB_10)
----------------------------------------
✅ 抽樣完成！已經成功建立包含 1000 筆資料的小型 HDF5 檔案。
📁 檔案位置: sample_1000.hdf5
----------------------------------------
